In [ ]:
import pandas as pd
import numpy as np

# Loading dataset

In [ ]:
data=pd.read_csv(r"C:\Users\RatanBiswakarmakar\Downloads\binance.csv")

In [ ]:
data

In [ ]:
#data = df.copy()

# Feature engineering

In [ ]:
def calculate_obv(data):
    obv = [0]

    for i in range(1, len(data)):
        if data['close'][i] > data['close'][i-1]:  
            obv.append(obv[-1] + data['volume'][i])
        elif data['close'][i] < data['close'][i-1]:  
            obv.append(obv[-1] - data['volume'][i])
        else:  
            obv.append(obv[-1])

    return obv

data['OBV'] = calculate_obv(data)
data.head()


In [ ]:
def calculate_rsi(data, period=14):
    delta = data['close'].diff()

    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)

    avg_gain = gain.rolling(window=period, min_periods=1).mean()
    avg_loss = loss.rolling(window=period, min_periods=1).mean()

    rs = avg_gain / avg_loss

    rsi = 100 - (100 / (1 + rs))

    return rsi

data['RSI'] = calculate_rsi(data)
data.head()


In [ ]:
def calculate_fibonacci_levels(data, period=14):
    rolling_high = data['high'].rolling(window=period, min_periods=1).max()
    rolling_low = data['low'].rolling(window=period, min_periods=1).min()

    data['Fib_23.6%'] = rolling_low + (rolling_high - rolling_low) * 0.236
    data['Fib_38.2%'] = rolling_low + (rolling_high - rolling_low) * 0.382
    data['Fib_50%'] = rolling_low + (rolling_high - rolling_low) * 0.5
    data['Fib_61.8%'] = rolling_low + (rolling_high - rolling_low) * 0.618

    return data

data = calculate_fibonacci_levels(data)
data.head()


In [ ]:
def calculate_stochastic_oscillator(data, k_period=14, d_period=3):
    low_min = data['low'].rolling(window=k_period).min()
    high_max = data['high'].rolling(window=k_period).max()

    data['%K'] = ((data['close'] - low_min) / (high_max - low_min)) * 100

    data['%D'] = data['%K'].rolling(window=d_period).mean()

    return data

data = calculate_stochastic_oscillator(data)
data.head()


In [ ]:
def calculate_parabolic_sar(data):
    data['SAR'] = 0.0
    data['EP'] = 0.0  
    data['AF'] = 0.02  

    # Initial trend
    uptrend = data['close'][1] > data['close'][0]

    # Initial EP
    if uptrend:
        data.at[0, 'EP'] = data['high'][0]
    else:
        data.at[0, 'EP'] = data['low'][0]

    # Initial SAR
    data.at[1, 'SAR'] = data['EP'][0]

    for i in range(2, len(data)):
        if uptrend:
            data.at[i, 'SAR'] = data['SAR'][i - 1] + data['AF'][i - 1] * (data['EP'][i - 1] - data['SAR'][i - 1])
        else:
            data.at[i, 'SAR'] = data['SAR'][i - 1] - data['AF'][i - 1] * (data['SAR'][i - 1] - data['EP'][i - 1])

        if uptrend:
            data.at[i, 'SAR'] = min(data.at[i, 'SAR'], data['low'][i - 1], data['low'][i])
            new_high = data['high'][i] > data['EP'][i - 1]
            data.at[i, 'EP'] = max(data['high'][i], data['EP'][i - 1])
        else:
            data.at[i, 'SAR'] = max(data.at[i, 'SAR'], data['high'][i - 1], data['high'][i])
            new_low = data['low'][i] < data['EP'][i - 1]
            data.at[i, 'EP'] = min(data['low'][i], data['EP'][i - 1])

        if (uptrend and new_high) or (not uptrend and new_low):
            data.at[i, 'AF'] = min(data['AF'][i - 1] + 0.02, 0.2)
        else:
            if (uptrend and data['low'][i] < data['SAR'][i]) or (not uptrend and data['high'][i] > data['SAR'][i]):
                uptrend = not uptrend
                data.at[i, 'SAR'] = data['EP'][i - 1]
                data.at[i, 'AF'] = 0.02
                data.at[i, 'EP'] = data['high'][i] if uptrend else data['low'][i]
            else:
                data.at[i, 'AF'] = data['AF'][i - 1]
                data.at[i, 'EP'] = data['EP'][i - 1]

    return data

data = calculate_parabolic_sar(data)
data.head()


In [ ]:
def calculate_adx(data, period=14):
    plus_dm = data['high'].diff()
    minus_dm = data['low'].diff()
    plus_dm[plus_dm < 0] = 0
    minus_dm[minus_dm > 0] = 0
    minus_dm = minus_dm.abs()

    tr1 = data['high'] - data['low']
    tr2 = (data['high'] - data['close'].shift()).abs()
    tr3 = (data['low'] - data['close'].shift()).abs()
    tr = pd.DataFrame({'tr1': tr1, 'tr2': tr2, 'tr3': tr3}).max(axis=1)

    smooth_plus_dm = plus_dm.rolling(window=period).sum()
    smooth_minus_dm = minus_dm.rolling(window=period).sum()
    smooth_tr = tr.rolling(window=period).sum()

    data['+DI'] = (smooth_plus_dm / smooth_tr) * 100
    data['-DI'] = (smooth_minus_dm / smooth_tr) * 100

    dx = (abs(data['+DI'] - data['-DI']) / (data['+DI'] + data['-DI'])) * 100
    data['ADX'] = dx.rolling(window=period).mean()

    return data

data = calculate_adx(data)
data.head()


In [ ]:
def calculate_macd(data, short_period=12, long_period=26, signal_period=9):
    short_ema = data['close'].ewm(span=short_period, adjust=False).mean()
    long_ema = data['close'].ewm(span=long_period, adjust=False).mean()

    data['MACD'] = short_ema - long_ema

    data['Signal Line'] = data['MACD'].ewm(span=signal_period, adjust=False).mean()

    data['MACD Histogram'] = data['MACD'] - data['Signal Line']

    return data

data = calculate_macd(data)
data.head()


In [ ]:
def calculate_bollinger_bands(data, period=20):
    data['Middle Band'] = data['close'].rolling(window=period).mean()

    std_dev = data['close'].rolling(window=period).std()

    data['Upper Band'] = data['Middle Band'] + (std_dev * 2)
    data['Lower Band'] = data['Middle Band'] - (std_dev * 2)

    return data

data = calculate_bollinger_bands(data)
data.head()


In [ ]:
data.columns

# Create labels Buy or Sell

In [ ]:
# Create labels based on future price movements
data['signal'] = 0
data['signal'][data['close'].shift(-1) > data['close']] = 1  # Buy signal
data['signal'][data['close'].shift(-1) < data['close']] = 0 

In [ ]:
data["signal"]

In [ ]:
data.columns

# Split the data

In [ ]:
from sklearn.model_selection import train_test_split

# Define features and labels
X = data.drop(columns=["signal","datetime","symbol"])
y = data['signal']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


# Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Create the model
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the model
model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# # Predict on the test set
# y_pred = model.predict(X_test)

# # Evaluate the model
# print(classification_report(y_test, y_pred))
# print(confusion_matrix(y_test, y_pred))

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

In [ ]:
data.isnull().sum()

In [ ]:
data.dropna(inplace=True)

In [ ]:
# Define the features (excluding 'datetime', 'symbol', 'future_return', 'signal')
features = ['open', 'high', 'low', 'close', 'volume', 'OBV', 'RSI', 'Fib_23.6%', 'Fib_38.2%', 
            'Fib_50%', 'Fib_61.8%', '%K', '%D', 'SAR', 'EP', 'AF', '+DI', '-DI', 'ADX', 'MACD', 
            'Signal Line', 'MACD Histogram', 'Middle Band', 'Upper Band', 'Lower Band']


In [ ]:
data.columns

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier,AdaBoostClassifier,GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import xgboost as xgb
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.svm import SVC


X = data.drop(columns=["signal","datetime","symbol"])
y = data["signal"]

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=343)

pipeline_lr = Pipeline([('scaler',StandardScaler()),
                    ('classifier',LogisticRegression())
                    ])
pipeline_rf = Pipeline([('scaler',StandardScaler()),
                    ('classifier',RandomForestClassifier(n_estimators=100,min_samples_split=10,max_depth=3))
                    ])
pipeline_xg = Pipeline([('scaler',StandardScaler()),
                    ('classifier',xgb.XGBClassifier(n_estimators=1000,learning_rate=0.09,max_depth=4,gamma=1,min_child_weight=1))
                    ])

pipeline_dt = Pipeline([('scaler',StandardScaler()),
                    ('classifier',DecisionTreeClassifier(criterion="gini",max_depth=4,min_samples_split=5))
                    ])

# pipeline_Bg = Pipeline([('scaler',StandardScaler()),
#     ('classifier',BaggingClassifier(n_estimators=100,max_samples=0.8,max_features=10))
#                     ])

pipeline_Bg = BaggingClassifier(n_estimators=1000,max_samples=0.8,max_features=10)
pipeline_svm = Pipeline([('scaler',StandardScaler()),
                    ('classifier',SVC(kernel="rbf",gamma=0.001))
                    ])
pipeline_ab = Pipeline([('scaler',StandardScaler()),
                    ('classifier',AdaBoostClassifier(n_estimators=100,learning_rate=0.09))
                    ])
# pipeline_gb = Pipeline([('scaler',StandardScaler()),
#                     ('classifier',GradientBoostingClassifier(n_estimators=1000,learning_rate=0.09,max_depth=4,min_samples_split=5))
#                     ])
pipeline_lr.fit(X_train,y_train)
print("Logistic Regression is running")
y_pred_test = pipeline_lr.predict(X_test)
print("Logistic Regression Classification Report Below!!!")
print(classification_report(y_test,y_pred_test))
print("Logistic Regression Pipeline is ended")

print("Random Forest Starts Now!!!!!!!!!!!")
pipeline_rf.fit(X_train,y_train)
print("Random Forestis running")
y_pred_test = pipeline_rf.predict(X_test)
print("Random Forest classification Report Below!!!!")
print(classification_report(y_test,y_pred_test))

print("xgboost is running")
pipeline_xg.fit(X_train,y_train)
y_pred_test =pipeline_xg.predict(X_test)
print("xgboost classification Report Below!!!!")
print(classification_report(y_test,y_pred_test))

print("decision tree is running")
pipeline_dt.fit(X_train,y_train)
y_pred_test = pipeline_dt.predict(X_test)
print("decision classification Report Below!!!!")
print(classification_report(y_test,y_pred_test))

print("Bagging is running")
pipeline_Bg.fit(X_train,y_train)
y_pred_test = pipeline_Bg.predict(X_test)
print("Bagging classification Report Below!!!!")
print(classification_report(y_test,y_pred_test))

print("SVM is running")
pipeline_svm.fit(X_train,y_train)
y_pred_test = pipeline_svm.predict(X_test)
print("SVM classification Report Below!!!!")
print(classification_report(y_test,y_pred_test))

print("Adaboosting is running")
pipeline_ab.fit(X_train,y_train)
y_pred_test = pipeline_ab.predict(X_test)
print("Adaboosting classification Report Below!!!!")
print(classification_report(y_test,y_pred_test))

# print("Gradient is running")
# pipeline_gb.fit(X_train,y_train)
# y_pred_test = pipeline_gb.predict(X_test)
# print("Gradientboosting classification Report Below!!!!")
# print(classification_report(y_test,y_pred_test))

In [ ]:
data.isnull().sum()

In [ ]:
data.columns

In [ ]:
# Define the function to calculate future return
def calculate_future_return(last_row):
    future_return_value = (last_row['close'] * 1.01) - last_row['close']
    return future_return_value / last_row['close']  # Percentage return

# Create new data by modifying existing values
last_row = data.iloc[-1]
new_data = last_row.copy()

In [ ]:
new_data['future_return'] = calculate_future_return(new_data)

# Convert new_data Series back to DataFrame
new_data_df = pd.DataFrame([new_data])



# Calculate future return using the defined function
new_data['future_return'] = calculate_future_return(new_data)
data = pd.concat([data, new_data_df], ignore_index=True)

In [ ]:
data.columns

In [ ]:
data.head(2)

# generate buy/sell signals on new data.

In [ ]:
features = ['open', 'high', 'low', 'close', 'volume', 'OBV',
       'RSI', 'Fib_23.6%', 'Fib_38.2%', 'Fib_50%', 'Fib_61.8%', '%K', '%D',
       'SAR', 'EP', 'AF', '+DI', '-DI', 'ADX', 'MACD', 'Signal Line',
       'MACD Histogram', 'Middle Band', 'Upper Band', 'Lower Band',
       'future_return', 'signal']  # Define your features
X = data[features]
y = np.where(data['future_return'] > 0, 1, -1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train the model (e.g., MLPClassifier)
model = MLPClassifier(hidden_layer_sizes=(50,), max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

# Predict on test set
y_pred = model.predict(X_test_scaled)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy * 100:.2f}%")

# Example of creating new data (modifying the last row for prediction)
new_data = X.iloc[-1].copy()  # Copy last row
new_data['open'] *= 1.01  # Adjust open price
new_data['high'] *= 1.01  # Adjust high price
new_data['low'] *= 0.99   # Adjust low price
new_data['close'] *= 1.01  # Adjust close price
new_data['volume'] *= 1.05  # Adjust volume

# Convert new data to a DataFrame and scale it
new_data_df = pd.DataFrame([new_data])
new_data_scaled = scaler.transform(new_data_df)

# Use the trained model to generate a signal on new data
signal = model.predict(new_data_scaled)

# Output the generated signal
print("Generated signal (1=Buy, -1=Sell):", signal)

In [ ]:
new_data_df.shape

In [ ]:
new_data_scaled